# Backtest Optimization Results

**Objective**: Visualize parameter optimization results from walk-forward validation

**Data Source**: `data/backtest/optimization_results.csv` (generated by `scripts/backtest_optimization.py`)

**Analysis**:
1. Sharpe Heatmaps (entry × exit percentile, faceted by sizing mode)
2. Stability Plot (train vs test Sharpe)
3. Degradation Analysis (histogram of test/train ratio)
4. Top 10 Configurations Table
5. Parameter Sensitivity (boxplots)

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Plotting style
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Load Optimization Results

In [ ]:
# Load CSV results
results_path = Path('../data/backtest/optimization_results.csv')
if not results_path.exists():
    print(f"ERROR: Results file not found: {results_path}")
    print("Run optimization first: python scripts/backtest_optimization.py")
    sys.exit(1)

df = pd.read_csv(results_path)

print(f"Loaded {len(df)} parameter combinations")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Load top 10 configs from JSON
json_path = Path('../data/backtest/best_params.json')
if json_path.exists():
    with open(json_path, 'r') as f:
        best_params = json.load(f)
    
    print(f"Optimization Metadata:")
    print(f"  Timestamp: {best_params['timestamp']}")
    print(f"  Train Period: {best_params['train_period']}")
    print(f"  Test Period: {best_params['test_period']}")
    print(f"  Universe Size: {best_params['universe_size']} tickers")
    print(f"  Total Combinations: {best_params['total_combinations']}")

## 2. Data Summary Statistics

In [ ]:
# Summary statistics
print("Training Period Metrics:")
print(df[['train_sharpe', 'train_calmar', 'train_max_dd', 'train_return', 'train_trades']].describe())

print("\nTest Period Metrics:")
print(df[['test_sharpe', 'test_calmar', 'test_max_dd', 'test_return', 'test_trades']].describe())

print("\nStability Metrics:")
print(df[['degradation', 'stability_score']].describe())

## 3. Sharpe Ratio Heatmaps

Visualize test Sharpe ratio across entry/exit percentile grid, faceted by sizing mode.

In [ ]:
# Create pivot table for each sizing mode
sizing_modes = df['sizing_mode'].unique()

fig, axes = plt.subplots(1, len(sizing_modes), figsize=(18, 5))
if len(sizing_modes) == 1:
    axes = [axes]

for i, mode in enumerate(sizing_modes):
    df_mode = df[df['sizing_mode'] == mode]
    
    # Pivot table (entry × exit)
    pivot = df_mode.pivot_table(
        index='entry_percentile',
        columns='exit_percentile',
        values='test_sharpe',
        aggfunc='mean'
    )
    
    # Heatmap
    sns.heatmap(
        pivot,
        annot=True,
        fmt='.2f',
        cmap='RdYlGn',
        center=0,
        cbar_kws={'label': 'Test Sharpe Ratio'},
        ax=axes[i]
    )
    
    axes[i].set_title(f'Sizing Mode: {mode}', fontsize=14, weight='bold')
    axes[i].set_xlabel('Exit Percentile', fontsize=12)
    axes[i].set_ylabel('Entry Percentile', fontsize=12)

plt.tight_layout()
plt.savefig('../data/backtest/sharpe_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved heatmap to: data/backtest/sharpe_heatmaps.png")

## 4. Stability Plot (Train vs Test Sharpe)

Scatter plot showing train Sharpe vs test Sharpe. Points near diagonal line = stable configs.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Scatter plot (color by sizing mode)
for mode in sizing_modes:
    df_mode = df[df['sizing_mode'] == mode]
    ax.scatter(
        df_mode['train_sharpe'],
        df_mode['test_sharpe'],
        label=mode,
        alpha=0.6,
        s=100
    )

# Diagonal reference line (perfect stability)
max_sharpe = max(df['train_sharpe'].max(), df['test_sharpe'].max())
ax.plot([0, max_sharpe], [0, max_sharpe], 'k--', alpha=0.3, label='Perfect Stability (1:1)')

# Degradation thresholds (80%, 60% lines)
ax.plot([0, max_sharpe], [0, max_sharpe * 0.8], 'r--', alpha=0.3, label='80% Degradation')
ax.plot([0, max_sharpe], [0, max_sharpe * 0.6], 'orange', linestyle='--', alpha=0.3, label='60% Degradation')

ax.set_xlabel('Train Sharpe Ratio', fontsize=14)
ax.set_ylabel('Test Sharpe Ratio', fontsize=14)
ax.set_title('Stability Plot: Train vs Test Sharpe Ratio', fontsize=16, weight='bold')
ax.legend(loc='upper left', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/backtest/stability_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved stability plot to: data/backtest/stability_plot.png")

## 5. Degradation Analysis

Histogram of degradation ratio (test/train Sharpe). Values near 1.0 = robust, <0.5 = overfitting.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Histogram
ax.hist(df['degradation'], bins=20, edgecolor='black', alpha=0.7)

# Reference lines
ax.axvline(1.0, color='green', linestyle='--', linewidth=2, label='Perfect Stability (1.0)')
ax.axvline(0.8, color='orange', linestyle='--', linewidth=2, label='Acceptable (0.8)')
ax.axvline(0.5, color='red', linestyle='--', linewidth=2, label='Overfitting (<0.5)')

# Median line
median_deg = df['degradation'].median()
ax.axvline(median_deg, color='blue', linestyle='-', linewidth=2, label=f'Median ({median_deg:.2f})')

ax.set_xlabel('Degradation Ratio (Test/Train Sharpe)', fontsize=14)
ax.set_ylabel('Frequency', fontsize=14)
ax.set_title('Degradation Analysis: Distribution of Test/Train Sharpe Ratio', fontsize=16, weight='bold')
ax.legend(loc='upper left', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/backtest/degradation_histogram.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Median degradation: {median_deg:.2f}")
print(f"Configs with degradation >= 0.8: {(df['degradation'] >= 0.8).sum()} / {len(df)} ({(df['degradation'] >= 0.8).mean() * 100:.1f}%)")

## 6. Top 10 Configurations Table

In [ ]:
# Sort by stability score, then test Sharpe
top_10 = df.sort_values(by=['stability_score', 'test_sharpe'], ascending=[False, False]).head(10)

# Display table
top_10_display = top_10[[
    'entry_percentile', 'exit_percentile', 'sizing_mode',
    'train_sharpe', 'test_sharpe', 'degradation', 'stability_score',
    'train_trades', 'test_trades'
]].reset_index(drop=True)

top_10_display.index += 1  # Start from 1
top_10_display.index.name = 'Rank'

print("\n" + "="*100)
print("TOP 10 STABLE CONFIGURATIONS")
print("="*100)
print(top_10_display.to_string())
print("="*100 + "\n")

# Save to CSV
top_10_display.to_csv('../data/backtest/top_10_configs.csv')
print("Saved top 10 table to: data/backtest/top_10_configs.csv")

## 7. Parameter Sensitivity Analysis

Boxplots showing test Sharpe distribution for each parameter value.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Entry percentile sensitivity
df.boxplot(column='test_sharpe', by='entry_percentile', ax=axes[0])
axes[0].set_title('Entry Percentile Sensitivity', fontsize=14, weight='bold')
axes[0].set_xlabel('Entry Percentile', fontsize=12)
axes[0].set_ylabel('Test Sharpe Ratio', fontsize=12)
axes[0].get_figure().suptitle('')  # Remove default title

# Exit percentile sensitivity
df.boxplot(column='test_sharpe', by='exit_percentile', ax=axes[1])
axes[1].set_title('Exit Percentile Sensitivity', fontsize=14, weight='bold')
axes[1].set_xlabel('Exit Percentile', fontsize=12)
axes[1].set_ylabel('Test Sharpe Ratio', fontsize=12)
axes[1].get_figure().suptitle('')

# Sizing mode sensitivity
df.boxplot(column='test_sharpe', by='sizing_mode', ax=axes[2])
axes[2].set_title('Sizing Mode Sensitivity', fontsize=14, weight='bold')
axes[2].set_xlabel('Sizing Mode', fontsize=12)
axes[2].set_ylabel('Test Sharpe Ratio', fontsize=12)
axes[2].get_figure().suptitle('')

plt.tight_layout()
plt.savefig('../data/backtest/parameter_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved sensitivity analysis to: data/backtest/parameter_sensitivity.png")

## 8. Robust Zone Identification

Identify "robust zone" where configs have:
- Degradation >= 0.8 (low performance drop)
- Test Sharpe >= 1.0 (acceptable out-of-sample performance)

In [ ]:
# Define robust zone criteria
robust_zone = df[
    (df['degradation'] >= 0.8) &
    (df['test_sharpe'] >= 1.0)
]

print(f"\n{'='*80}")
print(f"ROBUST ZONE ANALYSIS")
print(f"{'='*80}\n")
print(f"Criteria:")
print(f"  - Degradation >= 0.8 (max 20% performance drop)")
print(f"  - Test Sharpe >= 1.0 (acceptable OOS performance)\n")
print(f"Robust Configs: {len(robust_zone)} / {len(df)} ({len(robust_zone)/len(df)*100:.1f}%)\n")

if len(robust_zone) > 0:
    print("Robust Zone Statistics:")
    print(robust_zone[['entry_percentile', 'exit_percentile', 'sizing_mode']].describe())
    
    # Most common parameters in robust zone
    print("\nMost Common Parameters in Robust Zone:")
    print(f"  Entry Percentile: {robust_zone['entry_percentile'].mode().values[0]}")
    print(f"  Exit Percentile: {robust_zone['exit_percentile'].mode().values[0]}")
    print(f"  Sizing Mode: {robust_zone['sizing_mode'].mode().values[0]}")
else:
    print("WARNING: No configs meet robust zone criteria. Consider relaxing thresholds.")

print(f"\n{'='*80}\n")

## 9. Recommended Production Parameters

Based on top stable configuration.

In [ ]:
# Best config (rank 1)
best_config = df.sort_values(by=['stability_score', 'test_sharpe'], ascending=[False, False]).iloc[0]

print(f"\n{'='*80}")
print(f"RECOMMENDED PRODUCTION PARAMETERS")
print(f"{'='*80}\n")
print(f"Parameters:")
print(f"  entry_percentile_min: {best_config['entry_percentile']:.2f}")
print(f"  exit_percentile_max: {best_config['exit_percentile']:.2f}")
print(f"  exit_use_percentile: True")
print(f"  sizing_mode: '{best_config['sizing_mode']}'\n")
print(f"Expected Performance (Based on Test Period):")
print(f"  Sharpe Ratio: {best_config['test_sharpe']:.2f}")
print(f"  Calmar Ratio: {best_config['test_calmar']:.2f}")
print(f"  Max Drawdown: {best_config['test_max_dd']:.2f}%")
print(f"  Total Return: {best_config['test_return']:.2f}%")
print(f"  Win Rate: {best_config['test_win_rate']:.2f}%")
print(f"  Total Trades: {int(best_config['test_trades'])}\n")
print(f"Stability Metrics:")
print(f"  Degradation: {best_config['degradation']:.2f} (test/train Sharpe ratio)")
print(f"  Stability Score: {best_config['stability_score']:.2f}\n")
print(f"{'='*80}\n")

# Save recommendation to JSON
recommendation = {
    'recommended_parameters': {
        'entry_percentile_min': float(best_config['entry_percentile']),
        'exit_percentile_max': float(best_config['exit_percentile']),
        'exit_use_percentile': True,
        'sizing_mode': best_config['sizing_mode'],
    },
    'expected_performance': {
        'test_sharpe': float(best_config['test_sharpe']),
        'test_calmar': float(best_config['test_calmar']),
        'test_max_dd': float(best_config['test_max_dd']),
        'test_return': float(best_config['test_return']),
        'test_win_rate': float(best_config['test_win_rate']),
        'test_trades': int(best_config['test_trades']),
    },
    'stability': {
        'degradation': float(best_config['degradation']),
        'stability_score': float(best_config['stability_score']),
    },
    'metadata': {
        'analysis_date': pd.Timestamp.now().isoformat(),
        'train_period': best_params.get('train_period', 'Unknown'),
        'test_period': best_params.get('test_period', 'Unknown'),
    }
}

with open('../data/backtest/recommended_params.json', 'w') as f:
    json.dump(recommendation, f, indent=2)

print("Saved recommendation to: data/backtest/recommended_params.json")

## Summary

This notebook analyzed 75 parameter combinations from walk-forward validation and identified:

1. **Best Stable Configuration**: Top-ranked by stability score and test Sharpe
2. **Robust Zone**: Configs with degradation >= 0.8 and test Sharpe >= 1.0
3. **Parameter Sensitivity**: Which params drive performance most
4. **Overfitting Risk**: Configs with high train Sharpe but low test Sharpe

**Outputs**:
- `data/backtest/sharpe_heatmaps.png` - Performance across entry/exit grid
- `data/backtest/stability_plot.png` - Train vs test Sharpe scatter
- `data/backtest/degradation_histogram.png` - Overfitting distribution
- `data/backtest/parameter_sensitivity.png` - Boxplots per param
- `data/backtest/top_10_configs.csv` - Top 10 stable configs
- `data/backtest/recommended_params.json` - Production recommendation

**Next Steps**:
1. Validate recommended params on 2025 data (future walk-forward)
2. Run Monte Carlo simulation for confidence intervals
3. Update production strategy config with recommended params